In [ ]:
import plotly.graph_objects as go
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.decomposition import PCA
import pandas as pd
import plotly.express as px

In [ ]:
%cd ..

c:\Users\Kayned\Documents\GitHub\SLEP\research\test_version


In [ ]:
from TrainModel.train import training
training()

ModuleNotFoundError: No module named 'torch'

In [ ]:
logs = np.load("Models/Checkpoints/training_logs.npz", allow_pickle=True)
ref = np.load("Models/Checkpoints/reference_embeddings.npz", allow_pickle=True)

In [ ]:
best_epoch = int(logs["best_epoch"])
best_val_loss = float(logs["best_val_loss"])
best_val_accuracy = float(logs["best_val_accuracy"])
best_train_dist = logs["best_train_dists"]
best_val_dist = logs["best_val_dists"]

print(f"Best epoch: {best_epoch}")
print(f"Best val loss: {best_val_loss:.4f}")
print(f"Best val accuracy: {best_val_accuracy:.2%}")
print()
print(f"Best train distance:")
print(f"Positive: {best_train_dist[0]}")
print(f"Negative: {best_train_dist[1]}")
print(f"Difference: {best_train_dist[2]}")
print()
print(f"Best validation distance:")
print(f"Positive: {best_val_dist[0]}")
print(f"Negative: {best_val_dist[1]}")
print(f"Difference: {best_val_dist[2]}")


Best epoch: 264
Best val loss: 0.6815
Best val accuracy: 80.30%

Best train distance:
Positive: 0.38064199686050415
Negative: 1.3945863246917725
Difference: 1.013944387435913

Best validation distance:
Positive: 0.6275914907455444
Negative: 1.38671875
Difference: 0.7591272592544556


In [ ]:
epochs = logs["epochs"]
train_loss = logs["train_loss"]
val_loss = logs["val_loss"]
val_acc = logs["val_acc"]

fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=train_loss, mode="lines", name="Train Loss"))
fig.add_trace(go.Scatter(x=epochs, y=val_loss, mode="lines", name="Val Loss"))
fig.add_trace(go.Scatter(x=epochs, y=val_acc, mode="lines", name="Val Accuracy"))

fig.update_layout(
    title="LSTM Training Metrics over Epochs",
    xaxis_title="Epochs",
    yaxis_title="Values",
    hovermode="x unified"
)

fig.show()

In [ ]:
y_true = logs["best_labels"]
y_pred = logs["best_prediction"]

labels_names = ref["id_to_label"]
labels_ids = np.arange(len(labels_names))

cm = confusion_matrix(y_true, y_pred, labels=labels_ids)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig = go.Figure(data=go.Heatmap(
    z = cm_norm,
    zmin = 0,
    zmax = 1,
    x = labels_names,
    y = labels_names,
    colorscale="turbo",
    text=cm,
    texttemplate="%{text}",
    hovertemplate="True: %{y}<br>Pred: %{x}<extra></extra>"
))

fig.update_layout(
    title="Prediction in Best Epoch",
    xaxis_title="Predicted label",
    yaxis_title="True label",
    width=1200,
    height=1000,
    margin=dict(l=150, r=50, t=80, b=150)
)

fig.show()

In [ ]:
loss_gap = val_loss - train_loss

fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=loss_gap, mode="lines", name="Val Loss - Train Loss"))

fig.update_layout(
    title="Loss Gap over Epochs",
    xaxis_title="Epochs",
    yaxis_title="Loss gap",
    hovermode="x unified"
)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred,
    labels=labels_ids,
    target_names=labels_names,
    zero_division=0
))

              precision    recall  f1-score   support

  BASKETBALL       1.00      0.77      0.87        13
         BEE       0.69      0.75      0.72        12
   BREAKFAST       0.73      0.67      0.70        12
        CALL       1.00      0.67      0.80         9
         CAR       0.80      0.89      0.84         9
       CAT 3       1.00      0.78      0.88         9
   CHRISTMAS       0.71      0.42      0.53        12
        DEAF       0.67      0.83      0.74        12
         DOG       0.65      0.79      0.71        14
         EAT       0.80      1.00      0.89        12
     FIREMAN       0.89      0.89      0.89         9
        FISH       0.88      0.78      0.82         9
    FOOTBALL       1.00      1.00      1.00         9
       GREEN       0.86      0.60      0.71        10
        HEAD       0.82      1.00      0.90         9
    HOSPITAL       0.73      0.67      0.70        12
         KID       0.90      1.00      0.95         9
       LUNCH       0.31    

In [ ]:
from sklearn.metrics import recall_score

recalls = recall_score(
    y_true,
    y_pred,
    labels=labels_ids,
    average=None,
    zero_division=0
)

order = np.argsort(recalls)

fig = go.Figure(go.Bar(
    x=recalls[order],
    y=labels_names[order],
    orientation="h"
))

fig.update_layout(
    title="Recall per Class",
    xaxis_title="Recall",
    yaxis_title="Class",
    width=1000,
    height=800,
)

fig.show()

In [ ]:
train_dists_mean = logs["train_dists"]
val_dists_mean = logs["val_dists"]

fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=train_dists_mean[:, 0], mode="lines", name="Train Positive Distance"))
fig.add_trace(go.Scatter(x=epochs, y=train_dists_mean[:, 1], mode="lines", name="Train Negative Distance"))
fig.add_trace(go.Scatter(x=epochs, y=val_dists_mean[:, 0], mode="lines", name="Val Positive Distance"))
fig.add_trace(go.Scatter(x=epochs, y=val_dists_mean[:, 1], mode="lines", name="Val Negative Distance"))

fig.update_layout(
    title="Embedding Distances over Epochs",
    xaxis_title="Epochs",
    yaxis_title="Distance",
    hovermode="x unified",
)

fig.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=train_dists_mean[:, 2], mode="lines", name="Train Difference"))
fig.add_trace(go.Scatter(x=epochs, y=val_dists_mean[:, 2], mode="lines", name="Val Difference"))

fig.update_layout(
    title="Embedding Distances between Positive and Negative Samples over Epochs",
    xaxis_title="Epochs",
    yaxis_title="Distance",
    hovermode="x unified",
)

fig.show()

In [ ]:
lr = logs["lr"]

fig = go.Figure()

fig.add_trace(go.Scatter(x=epochs, y=lr, mode="lines", name="Train Loss"))

fig.update_layout(
    title="Learning Rate over Epochs",
    xaxis_title="Epochs",
    yaxis_title="Learnig Rate",
    hovermode="x unified",
)

fig.show()

In [ ]:
from sklearn.manifold import TSNE

embeddings = ref["embeddings"]
labels = ref["labels"]
id_to_label = ref["id_to_label"]

class_names = [id_to_label[label] for label in labels]

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=42
)

embeddings_2d = tsne.fit_transform(embeddings)

df = pd.DataFrame({
    "TSNE1": embeddings_2d[:, 0],
    "TSNE2": embeddings_2d[:, 1],
    "Class": class_names
})

fig = px.scatter(
    df,
    x="TSNE1",
    y="TSNE2",
    color="Class",
    title="t-SNE of Gesture Embeddings",
    width=1000,
    height=1000
)

fig.show()